In [1]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlalchemy

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

logging.basicConfig(
    filename= "Data Analysis Projects//Vendor Performance//logs/ingestion_db.log",
    level= logging.DEBUG,
    format= "%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name,engine):
    df.to_sql(table_name, con = engine, if_exists = 'replace', index = False, chunksize = 10000)

def load_raw_data():
    '''this function will load the CSVs as dataframe and ingest into db'''
    start = time.time()
    for file in os.listdir('Data Analysis Projects//Vendor Performance//data//data'):
        if'.csv' in file:
            df = pd.read_csv('Data Analysis Projects//Vendor Performance//data//data/'+file)
            logging.info(f'ingesting {file} in db')
            ingest_db(df, file[:-4], engine)
    end = time.time()
    total_time = (end - start)/60
    logging.info('-------------Ingestion Complete-------------')
    logging.info(f'Total Time Taken: {total_time} minutes')  

if __name__ == '__main__':
    load_raw_data()
    

## EXPLORATORY DATA ANALYSIS


In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('inventory.db')

In [4]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'",conn)
tables

,name
0,begin_inventory
1,end_inventory
2,purchases
3,purchase_prices
4,sales
5,vendor_invoice


In [5]:
pd.read_sql("SELECT COUNT(*) FROM purchases",conn)

,COUNT(*)
0,2372474


In [7]:
purchases = pd.read_sql("SELECT * FROM purchases WHERE VendorNumber = 4466",conn)
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-07,2024-01-19,2024-02-26,9.41,6,56.46,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-05,2024-01-19,2024-02-26,9.41,6,56.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2187,81_PEMBROKE_5215,81,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-29,2025-01-04,2025-02-10,9.41,6,56.46,1
2188,62_KILMARNOCK_5255,62,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.35,5,46.75,1
2189,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.41,5,47.05,1
2190,6_GOULCREST_5215,6,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-31,2025-01-04,2025-02-10,9.41,6,56.46,1


In [8]:
purchases.groupby(['Brand','PurchasePrice']) [['Quantity', 'Dollars']].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,4640,51921.60
5215,9.41,4923,46325.43
5255,9.35,6215,58110.25


In [5]:
purchase_prices = pd.read_sql_query("""SELECT * FROM purchase_prices where vendornumber = 4466""",conn)
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [6]:
vendor_invoice = pd.read_sql_query("""SELECT * FROM vendor_invoice WHERE VendorNumber = 4466""",conn)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.87,6.61,None


In [6]:
vendor_invoice['PONumber'].nunique()

55

In [3]:
sales = pd.read_sql("SELECT * FROM sales WHERE VendorNo = 4466",conn)
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


In [11]:
sales.groupby('Brand')[['SalesDollars','SalesPrice','SalesQuantity']].sum()

,SalesDollars,SalesPrice,SalesQuantity
Brand,,,
3140,50531.10,30071.85,3890
5215,60416.49,41542.02,4651
5255,79187.04,51180.60,6096


In [12]:
vendor_invoice.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')

In [4]:
freight_summary = pd.read_sql_query("""SELECT vendornumber, SUM(Freight) as FreightCost 
FROM vendor_invoice
GROUP BY VendorNumber""", conn)
freight_summary

,VendorNumber,FreightCost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19
...,...,...
121,98450,856.02
122,99166,130.09
123,172662,178.34
124,173357,202.50


In [6]:
pd.read_sql_query("""
SELECT p.VendorNumber, 
p.VendorName,
p.Brand, 
p.PurchasePrice, 
pp.Volume, 
pp.Price as ActualPrice,
SUM(p.Quantity) AS TotalPurchaseQuantity,
SUM(p.dollars) AS TotalPurchaseDollars
FROM purchases p 
JOIN purchase_prices pp
ON p.Brand = pp.Brand
WHERE p.purchasePrice > 0
GROUP BY p.VendorNumber, p.VendorName, p.Brand
ORDER BY TotalPurchaseDollars
""", conn)

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars
0,7245,PROXIMO SPIRITS INC.,3065,0.71,50,0.99,1,0.71
1,3960,DIAGEO NORTH AMERICA INC,6127,1.47,200,1.99,1,1.47
2,3924,HEAVEN HILL DISTILLERIES,9123,0.74,50,0.99,2,1.48
3,8004,SAZERAC CO INC,5683,0.39,50,0.49,6,2.34
4,9815,WINE GROUP INC,8527,1.32,750,4.99,2,2.64
...,...,...,...,...,...,...,...,...
10687,3960,DIAGEO NORTH AMERICA INC,3545,21.89,1750,29.99,138109,3023206.01
10688,3960,DIAGEO NORTH AMERICA INC,4261,16.17,1750,22.99,201682,3261197.94
10689,17035,PERNOD RICARD USA,8068,18.24,1750,24.99,187407,3418303.68
10690,4425,MARTIGNETTI COMPANIES,3405,23.19,1750,28.99,164038,3804041.22


In [9]:
sales.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='object')

In [12]:
pd.read_sql_query("""
SELECT VendorNo,
Brand, 
SUM(SalesDollars) AS TotalSalesDollars,
SUM(SalesPrice) AS TotalSalesPrice, 
SUM(SalesQuantity) AS TotalSalesQuantity, 
SUM(ExciseTax) AS TotalExciseTax
FROM sales
GROUP BY VendorNo, Brand
""", conn)

,VendorNo,Brand,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax
0,2,90085,665.82,295.92,18,2.00
1,2,90609,599.76,449.82,24,0.52
2,60,771,704.53,494.67,47,37.01
3,60,3979,66871.69,41682.51,3931,7224.06
4,105,2529,359.88,59.98,12,9.44
...,...,...,...,...,...,...
11267,173357,2804,6298.60,3194.29,140,110.33
11268,173357,3666,8996.40,4873.05,360,141.19
11269,173357,3848,185.94,92.97,6,4.71
11270,173357,3909,24540.18,14469.21,982,773.87


In [5]:
vendor_summary_table = pd.read_sql_query("""
WITH FreightSummary AS (
    SELECT 
        VendorNumber, 
        SUM(Freight) AS TotalFreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),
PurchaseSummary AS (
    SELECT 
        p.VendorNumber, 
        p.VendorName,
        p.Brand, 
        p.description,
        p.PurchasePrice, 
        pp.Volume, 
        pp.Price as ActualPrice, 
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.dollars) AS TotalPurchaseDollars
    FROM purchases p 
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.purchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.description, p.PurchasePrice, pp.Volume, pp.Price
),
SalesSummary AS (
    SELECT 
        VendorNo,
        Brand, 
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice, 
        SUM(SalesQuantity) AS TotalSalesQuantity, 
        SUM(ExciseTax) AS TotalExciseTax -- Ensure column name 'TotalExciseTax' exists in sales table
    FROM sales
    GROUP BY VendorNo, Brand
)
SELECT 
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description, 
    ps.ActualPrice,
    ps.PurchasePrice,
    ps.Volume,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice, 
    ss.TotalSalesQuantity, 
    ss.TotalExciseTax,
    fs.TotalFreightCost
FROM PurchaseSummary ps 
LEFT JOIN SalesSummary ss 
    ON ps.VendorNumber = ss.VendorNo
    AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber
ORDER BY ps.TotalPurchaseDollars DESC 
""", conn)

In [6]:
vendor_summary_table

,VendorNumber,VendorName,Brand,description,ActualPrice,PurchasePrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,TotalFreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,36.99,26.27,1750,145080,3811251.60,5.101920e+06,672819.31,142049.0,260999.20,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,28.99,23.19,1750,164038,3804041.22,4.819073e+06,561512.37,160247.0,294438.66,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,24.99,18.24,1750,187407,3418303.68,4.538121e+06,461140.15,187140.0,343854.07,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,22.99,16.17,1750,201682,3261197.94,4.475973e+06,420050.01,200412.0,368242.80,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,29.99,21.89,1750,138109,3023206.01,4.223108e+06,545778.28,135838.0,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,4.99,1.32,750,2,2.64,1.595000e+01,10.96,5.0,0.55,27100.41
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.49,0.39,50,6,2.34,6.566000e+01,1.47,134.0,7.04,50293.62
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.99,0.74,50,2,1.48,1.980000e+00,0.99,2.0,0.10,14069.87
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.99,1.47,200,1,1.47,1.432800e+02,77.61,72.0,15.12,257032.07


In [15]:
vendor_summary_table.dtypes

VendorNumber               int64
VendorName                object
Brand                      int64
description               object
ActualPrice              float64
PurchasePrice            float64
Volume                    object
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalSalesQuantity       float64
TotalExciseTax           float64
TotalFreightCost         float64
dtype: object

In [16]:
vendor_summary_table.isnull().sum()

VendorNumber               0
VendorName                 0
Brand                      0
description                0
ActualPrice                0
PurchasePrice              0
Volume                     0
TotalPurchaseQuantity      0
TotalPurchaseDollars       0
TotalSalesDollars        178
TotalSalesPrice          178
TotalSalesQuantity       178
TotalExciseTax           178
TotalFreightCost           0
dtype: int64

In [17]:
vendor_summary_table['VendorName'].unique()

array(['BROWN-FORMAN CORP          ', 'MARTIGNETTI COMPANIES',
       'PERNOD RICARD USA          ', 'DIAGEO NORTH AMERICA INC   ',
       'BACARDI USA INC            ', 'JIM BEAM BRANDS COMPANY    ',
       'MAJESTIC FINE WINES        ', 'ULTRA BEVERAGE COMPANY LLP ',
       'STOLI GROUP,(USA) LLC      ', 'PROXIMO SPIRITS INC.       ',
       'MOET HENNESSY USA INC      ', 'CAMPARI AMERICA            ',
       'SAZERAC CO INC             ', 'CONSTELLATION BRANDS INC   ',
       'M S WALKER INC             ', 'SAZERAC NORTH AMERICA INC. ',
       'PALM BAY INTERNATIONAL INC ', 'REMY COINTREAU USA INC     ',
       'SIDNEY FRANK IMPORTING CO  ', 'E & J GALLO WINERY         ',
       'WILLIAM GRANT & SONS INC   ', 'HEAVEN HILL DISTILLERIES   ',
       'DISARONNO INTERNATIONAL LLC', 'EDRINGTON AMERICAS         ',
       'CASTLE BRANDS CORP.        ', 'SOUTHERN WINE & SPIRITS NE ',
       'STE MICHELLE WINE ESTATES  ', 'TRINCHERO FAMILY ESTATES   ',
       'MHW LTD                    ', 'W

In [18]:
vendor_summary_table['Volume'] = vendor_summary_table['Volume'].astype('float64')

In [19]:
vendor_summary_table.fillna(0, inplace = True)

In [20]:
vendor_summary_table['VendorName'] = vendor_summary_table['VendorName'].str.strip()

In [21]:
vendor_summary_table['GrossProfit'] = vendor_summary_table['TotalSalesDollars'] - vendor_summary_table['TotalPurchaseDollars']

In [23]:
vendor_summary_table['ProfitMargin'] = (vendor_summary_table['GrossProfit']/vendor_summary_table['TotalSalesDollars'])*100 

In [25]:
vendor_summary_table['StockTurnover'] = vendor_summary_table['TotalSalesQuantity']/vendor_summary_table['TotalPurchaseQuantity']

In [26]:
vendor_summary_table['SalesToPurchaseRatio'] = vendor_summary_table['TotalSalesDollars']/vendor_summary_table['TotalPurchaseDollars']

In [27]:
cursor = conn.cursor()

In [28]:
cursor.execute("""CREATE TABLE vendor_sales_summary(
VendorNumber INT,
VendorName VARCHAR(100),
Brand INT,
description VARCHAR(100),
PurchasePrice DECIMAL(10,2),
ActualPrice DECIMAL(10,2),
Volume, 
TotalPurchaseQuantity INT, 
TotalPurchaseDollars DECIMAL(15,2),
TotalSalesQuantity INT,
TotalSalesDollars DECIMAL(15,2),
TotalSalesPrice DECIMAL(15,2),
TotalExciseTax DECIMAL(15,2),
FreightCost DECIMAL(15,2), 
GrossProfit DECIMAL(15,2),
ProfitMargin  DECIMAL(15,2), 
StockTurnover  DECIMAL(15,2), 
SalesToPurchaseRatio  DECIMAL(15,2), 
PRIMARY KEY (VendorNumber, Brand)
);""")

In [32]:
pd.read_sql_query("SELECT * FROM vendor_sales_summary",conn)

,VendorNumber,VendorName,Brand,description,ActualPrice,PurchasePrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,TotalFreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,36.99,26.27,1750.0,145080,3811251.60,5.101920e+06,672819.31,142049.0,260999.20,68601.68,1290667.91,25.297693,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,28.99,23.19,1750.0,164038,3804041.22,4.819073e+06,561512.37,160247.0,294438.66,144929.24,1015032.27,21.062810,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,24.99,18.24,1750.0,187407,3418303.68,4.538121e+06,461140.15,187140.0,343854.07,123780.22,1119816.92,24.675786,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,22.99,16.17,1750.0,201682,3261197.94,4.475973e+06,420050.01,200412.0,368242.80,257032.07,1214774.94,27.139908,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,29.99,21.89,1750.0,138109,3023206.01,4.223108e+06,545778.28,135838.0,249587.83,257032.07,1199901.61,28.412764,0.983556,1.396897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,4.99,1.32,750.0,2,2.64,1.595000e+01,10.96,5.0,0.55,27100.41,13.31,83.448276,2.500000,6.041667
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.49,0.39,50.0,6,2.34,6.566000e+01,1.47,134.0,7.04,50293.62,63.32,96.436186,22.333333,28.059829
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.99,0.74,50.0,2,1.48,1.980000e+00,0.99,2.0,0.10,14069.87,0.50,25.252525,1.000000,1.337838
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.99,1.47,200.0,1,1.47,1.432800e+02,77.61,72.0,15.12,257032.07,141.81,98.974037,72.000000,97.469388


In [31]:
vendor_summary_table.to_sql('vendor_sales_summary',conn , if_exists = 'replace', index = False)

10692

In [7]:
import sqlite3
import pandas as pd 
import logging
import sys
import os

# Define the path to the folder where ingestion_db.py lives
module_path = os.path.abspath("Data Analysis Projects//Vendor Performance")

# Add that path to the system's search list
if module_path not in sys.path:
    sys.path.append(module_path)

from ingestion_db import ingest_db
 
logging.basicConfig(
    filename = "Data Analysis Projects//Vendor Performance//logs/get_vendor_summary.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"
)

def create_vendor_summary(conn):
    '''this function will merge the different tables to get the overall vendor summary and adding new columns in the resultant data'''
    vendor_summary_table = pd.read_sql_query("""
    WITH FreightSummary AS (
    SELECT 
        VendorNumber, 
        SUM(Freight) AS TotalFreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
    ),
    PurchaseSummary AS (
    SELECT 
        p.VendorNumber, 
        p.VendorName,
        p.Brand, 
        p.description,
        p.PurchasePrice, 
        pp.Volume, 
        pp.Price as ActualPrice, 
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.dollars) AS TotalPurchaseDollars
    FROM purchases p 
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.purchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.description, p.PurchasePrice, pp.Volume, pp.Price
    ),
    SalesSummary AS (
    SELECT 
        VendorNo,
        Brand, 
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice, 
        SUM(SalesQuantity) AS TotalSalesQuantity, 
        SUM(ExciseTax) AS TotalExciseTax -- Ensure column name 'TotalExciseTax' exists in sales table
    FROM sales
    GROUP BY VendorNo, Brand
    )
    SELECT 
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description, 
    ps.ActualPrice,
    ps.PurchasePrice,
    ps.Volume,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice, 
    ss.TotalSalesQuantity, 
    ss.TotalExciseTax,
    fs.TotalFreightCost
    FROM PurchaseSummary ps 
    LEFT JOIN SalesSummary ss 
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC 
    """, conn)
    return vendor_summary_table

def clean_data(df):
    '''this function will clean the data'''
    df['Volume'] = df['Volume'].astype('float64')
    df.fillna(0, inplace = True)
    df['VendorName'] = df['VendorName'].str.strip()
   
    vendor_summary_table['GrossProfit'] = vendor_summary_table['TotalSalesDollars'] - vendor_summary_table['TotalPurchaseDollars']
    vendor_summary_table['ProfitMargin'] = (vendor_summary_table['GrossProfit']/vendor_summary_table['TotalSalesDollars'])*100 
    vendor_summary_table['StockTurnover'] = vendor_summary_table['TotalSalesQuantity']/vendor_summary_table['TotalPurchaseQuantity']
    vendor_summary_table['SalesToPurchaseRatio'] = vendor_summary_table['TotalSalesDollars']/vendor_summary_table['TotalPurchaseDollars']
    return df

if __name__ == '__main__':
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table.....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data.....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info('Ingesting data.....')
    ingest_db(clean_df,'vendor_summary_table', conn)
    logging.info('Completed')

    